<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/ai/04_generative_ai/genai_projects/experiment_production_rag_pipeline_langchain_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# =============================================================================
# CELL 1 : INSTALL REQUIRED LIBRARIES
# =============================================================================

!pip -q install -U \
google-genai \
langchain-core \
langchain-text-splitters \
langchain-huggingface \
langchain-chroma \
chromadb \
sentence-transformers \
pypdf \
python-docx \
unstructured \
python-dotenv \
tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 23.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.0/958.0 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.3/252.3 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 29.1 MB/s eta 0:00:0

In [13]:
# =============================================================================
# CELL 3 : GEMINI IMPORTS
# =============================================================================

from google import genai

In [14]:
# =============================================================================
# CELL 4 : LANGCHAIN IMPORTS
# =============================================================================

from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_chroma import Chroma

In [15]:
# =============================================================================
# CELL 5 : DOCUMENT LOADERS
# =============================================================================

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    DirectoryLoader
)

In [16]:
# =============================================================================
# CELL 6 : LOGGING
# =============================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("ProductionRAG")

In [17]:
# =============================================================================
# CELL 7 : GOOGLE API KEY
# =============================================================================

from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("API Loaded")

API Loaded


In [18]:
# =============================================================================
# CELL 8 : GEMINI CLIENT
# =============================================================================

client = genai.Client(
    api_key=GOOGLE_API_KEY
)

In [19]:
# =============================================================================
# CELL 9 : PROJECT PATHS
# =============================================================================

BASE_DIR = Path("/content")

DATA_DIR = BASE_DIR / "documents"

VECTOR_DB = BASE_DIR / "vector_db"

DATA_DIR.mkdir(exist_ok=True)

VECTOR_DB.mkdir(exist_ok=True)

In [20]:
# =============================================================================
# CELL 10 : SAMPLE DATA
# =============================================================================

sample = """
RAG stands for Retrieval Augmented Generation.

The capital of France is Paris.

The capital of India is New Delhi.

Gemini is Google's family of AI models.

LangChain is an orchestration framework for LLM applications.
"""

with open(DATA_DIR/"sample.txt","w") as f:
    f.write(sample)

In [21]:
# =============================================================================
# CELL 11 : LOAD DOCUMENTS
# =============================================================================

loader = DirectoryLoader(

    DATA_DIR,

    glob="**/*.txt",

    loader_cls=TextLoader

)

documents = loader.load()

print(len(documents))

1


In [22]:
documents

[Document(metadata={'source': '/content/documents/sample.txt'}, page_content="\nRAG stands for Retrieval Augmented Generation.\n\nThe capital of France is Paris.\n\nThe capital of India is New Delhi.\n\nGemini is Google's family of AI models.\n\nLangChain is an orchestration framework for LLM applications.\n")]

In [23]:
# =============================================================================
# CELL 13 : TEXT SPLITTER
# =============================================================================

splitter = RecursiveCharacterTextSplitter(

    chunk_size=400,

    chunk_overlap=50

)

In [24]:
# =============================================================================
# CELL 14 : CREATE CHUNKS
# =============================================================================

chunks = splitter.split_documents(documents)

print(len(chunks))

1


In [25]:
for chunk in chunks:

    print("="*70)

    print(chunk.page_content)

RAG stands for Retrieval Augmented Generation.

The capital of France is Paris.

The capital of India is New Delhi.

Gemini is Google's family of AI models.

LangChain is an orchestration framework for LLM applications.


In [26]:
# =============================================================================
# CELL 16 : EMBEDDING MODEL
# =============================================================================

embedding_model = HuggingFaceEmbeddings(

    model_name="BAAI/bge-small-en-v1.5",

    model_kwargs={

        "device":"cpu"

    },

    encode_kwargs={

        "normalize_embeddings":True

    }

)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [27]:
# =============================================================================
# CELL 17 : VECTOR DATABASE
# =============================================================================

vector_db = Chroma.from_documents(

    documents=chunks,

    embedding=embedding_model,

    persist_directory=str(VECTOR_DB)

)

In [28]:
# =============================================================================
# CELL 18 : RETRIEVER
# =============================================================================

retriever = vector_db.as_retriever(

    search_type="similarity",

    search_kwargs={

        "k":3

    }

)

In [29]:
docs = retriever.invoke(

    "What is RAG?"

)

for d in docs:

    print(d.page_content)

    print()

RAG stands for Retrieval Augmented Generation.

The capital of France is Paris.

The capital of India is New Delhi.

Gemini is Google's family of AI models.

LangChain is an orchestration framework for LLM applications.



In [30]:
# =============================================================================
# CELL 20 : PROMPT FUNCTION
# =============================================================================

def build_prompt(question, retrieved_docs):

    context = "\n\n".join(

        doc.page_content

        for doc in retrieved_docs

    )

    prompt = f"""
You are an AI Assistant.

Answer ONLY from the provided context.

If the answer is unavailable,
reply

I don't know.

Context

{context}

Question

{question}

Answer
"""

    return prompt

In [31]:
# =============================================================================
# CELL 21 : RETRIEVE DOCUMENTS
# =============================================================================

def retrieve_documents(
    question: str,
    k: int = 3
):
    """
    Retrieve top-k relevant documents.

    Parameters
    ----------
    question : str
        User query.

    k : int
        Number of retrieved chunks.

    Returns
    -------
    List[Document]
    """

    retriever = vector_db.as_retriever(

        search_type="similarity",

        search_kwargs={

            "k": k

        }

    )

    docs = retriever.invoke(question)

    return docs

In [32]:
# =============================================================================
# CELL 22 : GENERATE ANSWER USING GEMINI
# =============================================================================

def generate_answer(question):

    docs = retrieve_documents(question)

    prompt = build_prompt(

        question,

        docs

    )

    response = client.models.generate_content(

        model="gemini-2.5-flash",

        contents=prompt

    )

    return response.text, docs

In [33]:
# =============================================================================
# CELL 23 : PRETTY OUTPUT
# =============================================================================

def print_response(

    question,

    answer,

    docs

):

    print("="*80)

    print("QUESTION")

    print(question)

    print()

    print("="*80)

    print("ANSWER")

    print(answer)

    print()

    print("="*80)

    print("SOURCES")

    for i,doc in enumerate(docs):

        print(f"\nSource {i+1}")

        print(doc.page_content)

In [34]:
# =============================================================================
# CELL 24 : ASK QUESTION
# =============================================================================

question = "What is RAG?"

answer, docs = generate_answer(question)

print_response(

    question,

    answer,

    docs

)

QUESTION
What is RAG?

ANSWER
RAG stands for Retrieval Augmented Generation.

SOURCES

Source 1
RAG stands for Retrieval Augmented Generation.

The capital of France is Paris.

The capital of India is New Delhi.

Gemini is Google's family of AI models.

LangChain is an orchestration framework for LLM applications.


In [35]:
question = "What is the capital of India?"

answer, docs = generate_answer(question)

print_response(

    question,

    answer,

    docs

)

QUESTION
What is the capital of India?

ANSWER
The capital of India is New Delhi.

SOURCES

Source 1
RAG stands for Retrieval Augmented Generation.

The capital of France is Paris.

The capital of India is New Delhi.

Gemini is Google's family of AI models.

LangChain is an orchestration framework for LLM applications.


In [36]:
question = "Who invented Linux?"

answer, docs = generate_answer(question)

print_response(

    question,

    answer,

    docs

)

QUESTION
Who invented Linux?

ANSWER
I don't know.

SOURCES

Source 1
RAG stands for Retrieval Augmented Generation.

The capital of France is Paris.

The capital of India is New Delhi.

Gemini is Google's family of AI models.

LangChain is an orchestration framework for LLM applications.


In [37]:
# =============================================================================
# CELL 27 : SHOW SIMILARITY SCORES
# =============================================================================

results = vector_db.similarity_search_with_score(

    "What is RAG?",

    k=3

)

for doc, score in results:

    print("="*80)

    print("Score :", score)

    print(doc.page_content)

Score : 0.5581974387168884
RAG stands for Retrieval Augmented Generation.

The capital of France is Paris.

The capital of India is New Delhi.

Gemini is Google's family of AI models.

LangChain is an orchestration framework for LLM applications.


In [38]:
# =============================================================================
# CELL 28 : DOCUMENT METADATA
# =============================================================================

for chunk in chunks:

    print(chunk.metadata)

{'source': '/content/documents/sample.txt'}


In [39]:
# =============================================================================
# CELL 29 : VECTOR DATABASE INFO
# =============================================================================

print("Documents :", len(documents))

print("Chunks :", len(chunks))

print("Embeddings :", vector_db._collection.count())

Documents : 1
Chunks : 1
Embeddings : 1


In [40]:
# =============================================================================
# CELL 30 : CHATBOT
# =============================================================================

while True:

    question = input("\nAsk : ")

    if question.lower() == "exit":

        print("Session Ended.")

        break

    answer, docs = generate_answer(question)

    print("\n")

    print(answer)


Ask : who is messi


I don't know.

Ask : exit
Session Ended.


In [41]:
# =============================================================================
# CELL 31 : CONFIGURATION
# =============================================================================

from dataclasses import dataclass

@dataclass
class Config:

    EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

    GEMINI_MODEL = "gemini-2.5-flash"

    CHUNK_SIZE = 500

    CHUNK_OVERLAP = 100

    TOP_K = 4

    DB_PATH = str(VECTOR_DB)

config = Config()

In [42]:
# =============================================================================
# CELL 32 : PRODUCTION SPLITTER
# =============================================================================

splitter = RecursiveCharacterTextSplitter(

    chunk_size=config.CHUNK_SIZE,

    chunk_overlap=config.CHUNK_OVERLAP,

    separators=[

        "\n\n",

        "\n",

        ".",

        " ",

        ""

    ]

)

In [43]:
# =============================================================================
# CELL 33 : EMBEDDING FACTORY
# =============================================================================

def get_embedding_model():

    return HuggingFaceEmbeddings(

        model_name=config.EMBEDDING_MODEL,

        model_kwargs={

            "device":"cpu"

        },

        encode_kwargs={

            "normalize_embeddings":True

        }

    )

In [44]:
# =============================================================================
# CELL 34 : LOAD PERSISTENT DATABASE
# =============================================================================

embedding_model = get_embedding_model()

vector_db = Chroma(

    persist_directory=config.DB_PATH,

    embedding_function=embedding_model

)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [45]:
# =============================================================================
# CELL 35 : INDEX DOCUMENTS
# =============================================================================

def index_documents(documents):

    chunks = splitter.split_documents(

        documents

    )

    vector_db.add_documents(

        chunks

    )

    print(f"Indexed {len(chunks)} chunks.")

In [46]:
# =============================================================================
# CELL 36 : LOAD ALL TXT FILES
# =============================================================================

loader = DirectoryLoader(

    DATA_DIR,

    glob="**/*.txt",

    loader_cls=TextLoader

)

documents = loader.load()

In [47]:
# =============================================================================
# CELL 37 : INDEX DATABASE
# =============================================================================

if vector_db._collection.count()==0:

    print("Creating Vector Database...")

    index_documents(documents)

else:

    print("Existing Database Loaded.")

Existing Database Loaded.


In [48]:
# =============================================================================
# CELL 38 : MMR RETRIEVER
# =============================================================================

retriever = vector_db.as_retriever(

    search_type="mmr",

    search_kwargs={

        "k":4,

        "fetch_k":20,

        "lambda_mult":0.7

    }

)

In [49]:
# =============================================================================
# CELL 39 : RETRIEVAL LOGGING
# =============================================================================

def retrieve(question):

    docs = retriever.invoke(

        question

    )

    logger.info(

        f"Retrieved {len(docs)} documents."

    )

    return docs

In [50]:
# =============================================================================
# CELL 40 : PROMPT
# =============================================================================

SYSTEM_PROMPT = """
You are an Enterprise AI Assistant.

Rules

1. Never hallucinate.

2. Answer ONLY from context.

3. If answer unavailable say

"I don't know."

4. Always be concise.

5. Cite evidence if possible.

"""

In [51]:
# =============================================================================
# CELL 41 : BUILD PROMPT
# =============================================================================

def build_prompt(question):

    docs = retrieve(question)

    context = "\n\n".join(

        d.page_content

        for d in docs

    )

    prompt = f"""

{SYSTEM_PROMPT}

Context

{context}

Question

{question}

Answer

"""

    return prompt,docs

In [52]:
# =============================================================================
# CELL 42 : GEMINI
# =============================================================================

def ask_llm(question):

    prompt,docs = build_prompt(question)

    response = client.models.generate_content(

        model=config.GEMINI_MODEL,

        contents=prompt

    )

    return response.text,docs

In [53]:
# =============================================================================
# CELL 43 : OUTPUT
# =============================================================================

def ask(question):

    answer,docs = ask_llm(question)

    print("="*80)

    print(question)

    print("="*80)

    print(answer)

    print("\nSources\n")

    for i,d in enumerate(docs):

        print(f"[{i+1}]")

        print(d.page_content)

        print()

In [54]:
ask("What is RAG?")

What is RAG?
RAG stands for Retrieval Augmented Generation.

Sources

[1]
RAG stands for Retrieval Augmented Generation.

The capital of France is Paris.

The capital of India is New Delhi.

Gemini is Google's family of AI models.

LangChain is an orchestration framework for LLM applications.



In [55]:
ask("What is Gemini?")

What is Gemini?
Gemini is Google's family of AI models. (Context)

Sources

[1]
RAG stands for Retrieval Augmented Generation.

The capital of France is Paris.

The capital of India is New Delhi.

Gemini is Google's family of AI models.

LangChain is an orchestration framework for LLM applications.



In [56]:
# =============================================================================
# CELL 46 : SUPPORTED FILE TYPES
# =============================================================================

SUPPORTED_EXTENSIONS = {
    ".txt",
    ".pdf",
    ".docx"
}

In [57]:
# =============================================================================
# CELL 47 : TXT LOADER
# =============================================================================

from langchain_community.document_loaders import TextLoader

def load_txt(path):

    loader = TextLoader(path)

    return loader.load()

In [58]:
# =============================================================================
# CELL 48 : PDF LOADER
# =============================================================================

from langchain_community.document_loaders import PyPDFLoader

def load_pdf(path):

    loader = PyPDFLoader(path)

    return loader.load()

In [59]:
# =============================================================================
# CELL 49 : DOCX LOADER
# =============================================================================

from langchain_community.document_loaders import Docx2txtLoader

def load_docx(path):

    loader = Docx2txtLoader(path)

    return loader.load()

In [60]:
# =============================================================================
# CELL 50 : GENERIC DOCUMENT LOADER
# =============================================================================

def load_document(file_path):

    ext = Path(file_path).suffix.lower()

    if ext == ".txt":
        return load_txt(file_path)

    elif ext == ".pdf":
        return load_pdf(file_path)

    elif ext == ".docx":
        return load_docx(file_path)

    else:
        raise ValueError(
            f"Unsupported File : {ext}"
        )

In [61]:
# =============================================================================
# CELL 51 : LOAD COMPLETE FOLDER
# =============================================================================

def load_folder(folder):

    all_docs = []

    folder = Path(folder)

    for file in folder.rglob("*"):

        if file.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue

        docs = load_document(str(file))

        all_docs.extend(docs)

    return all_docs

In [62]:
# =============================================================================
# CELL 52 : ADD METADATA
# =============================================================================

def enrich_metadata(documents):

    for doc in documents:

        source = doc.metadata.get("source","Unknown")

        doc.metadata["filename"] = Path(source).name

        doc.metadata["extension"] = Path(source).suffix

    return documents

In [63]:
# =============================================================================
# CELL 53 : LOAD DATASET
# =============================================================================

documents = load_folder(DATA_DIR)

documents = enrich_metadata(documents)

print(

    f"Loaded {len(documents)} Documents"

)

Loaded 1 Documents


In [64]:
# =============================================================================
# CELL 54 : VERIFY METADATA
# =============================================================================

documents[0].metadata

{'source': '/content/documents/sample.txt',
 'filename': 'sample.txt',
 'extension': '.txt'}

In [65]:
# =============================================================================
# CELL 55 : SPLIT DOCUMENTS
# =============================================================================

chunks = splitter.split_documents(

    documents

)

print(

    len(chunks)

)

1


In [66]:
# =============================================================================
# CELL 56 : VERIFY CHUNK METADATA
# =============================================================================

chunks[0].metadata

{'source': '/content/documents/sample.txt',
 'filename': 'sample.txt',
 'extension': '.txt'}

In [67]:
# =============================================================================
# CELL 57 : UPDATE VECTOR DATABASE
# =============================================================================

vector_db.add_documents(

    chunks

)

print(

    vector_db._collection.count()

)

2


In [68]:
# =============================================================================
# CELL 58 : FILTERING
# =============================================================================

results = vector_db.similarity_search(

    "RAG",

    k=3,

    filter={

        "extension":".txt"

    }

)

for r in results:

    print(r.metadata)

    print(r.page_content)

    print()

{'source': '/content/documents/sample.txt', 'extension': '.txt', 'filename': 'sample.txt'}
RAG stands for Retrieval Augmented Generation.

The capital of France is Paris.

The capital of India is New Delhi.

Gemini is Google's family of AI models.

LangChain is an orchestration framework for LLM applications.



In [69]:
# =============================================================================
# CELL 59 : FORMAT SOURCES
# =============================================================================

def format_sources(docs):

    sources = []

    for d in docs:

        name = d.metadata.get(

            "filename",

            "Unknown"

        )

        if name not in sources:

            sources.append(name)

    return sources

In [70]:
# =============================================================================
# CELL 60 : IMPROVED RESPONSE
# =============================================================================

def ask(question):

    answer,docs = ask_llm(question)

    print("="*80)

    print("QUESTION")

    print(question)

    print()

    print("="*80)

    print("ANSWER")

    print(answer)

    print()

    print("="*80)

    print("DOCUMENTS USED")

    for source in format_sources(docs):

        print("-",source)

In [71]:
# =============================================================================
# CELL 61 : PERFORMANCE TIMER
# =============================================================================

import time

In [72]:
# =============================================================================
# CELL 62 : QUERY EXPANSION
# =============================================================================

QUERY_EXPANSION_PROMPT = """
You are an AI assistant.

Generate 5 different search queries
that have the same meaning.

Question:

{question}

Only return the queries.

One per line.
"""

In [73]:
# =============================================================================
# CELL 63 : QUERY EXPANSION
# =============================================================================

def generate_queries(question):

    prompt = QUERY_EXPANSION_PROMPT.format(
        question=question
    )

    response = client.models.generate_content(
        model=config.GEMINI_MODEL,
        contents=prompt
    )

    queries = [
        q.strip()
        for q in response.text.split("\n")
        if q.strip()
    ]

    queries.insert(0, question)

    return queries

In [74]:
queries = generate_queries(
    "Explain RAG"
)

queries

['Explain RAG',
 'Explain RAG',
 'What is RAG?',
 'RAG explained',
 'How does RAG work?',
 'Understanding RAG concept']

In [75]:
# =============================================================================
# CELL 65 : MULTI QUERY RETRIEVAL
# =============================================================================

def retrieve_multi_query(question):

    queries = generate_queries(question)

    retrieved = []

    for q in queries:

        docs = retriever.invoke(q)

        retrieved.extend(docs)

    return retrieved

In [76]:
# =============================================================================
# CELL 66 : REMOVE DUPLICATES
# =============================================================================

def unique_documents(docs):

    unique = {}

    for doc in docs:

        unique[doc.page_content] = doc

    return list(unique.values())


In [77]:
docs = retrieve_multi_query(
    "Explain RAG"
)

docs = unique_documents(docs)

len(docs)

1

In [78]:
# =============================================================================
# CELL 68 : THRESHOLD FILTER
# =============================================================================

def retrieve_with_threshold(

    question,

    threshold=1.2

):

    results = vector_db.similarity_search_with_score(

        question,

        k=10

    )

    filtered = []

    for doc,score in results:

        if score < threshold:

            filtered.append(doc)

    return filtered

In [79]:
# =============================================================================
# CELL 69 : MEASURE RETRIEVAL
# =============================================================================

start = time.time()

docs = retrieve_multi_query(
    "Explain RAG"
)

end = time.time()

print(f"{end-start:.2f} seconds")

1.72 seconds


In [80]:
# =============================================================================
# CELL 70 : ADVANCED PROMPT
# =============================================================================

SYSTEM_PROMPT = """
You are an Enterprise AI Assistant.

Instructions

1. Never hallucinate.

2. Use ONLY the retrieved context.

3. If uncertain say

'I don't know.'

4. Answer using markdown.

5. Use bullet points whenever possible.

6. Cite filenames whenever available.

7. Keep answers factual.
"""

In [81]:
# =============================================================================
# CELL 71 : BUILD PROMPT
# =============================================================================

def build_prompt(question):

    docs = unique_documents(

        retrieve_multi_query(question)

    )

    context = "\n\n".join(

        d.page_content

        for d in docs

    )

    prompt = f"""

{SYSTEM_PROMPT}

Context

{context}

Question

{question}

Answer

"""

    return prompt,docs

In [82]:
# =============================================================================
# CELL 72 : GENERATE
# =============================================================================

def ask_llm(question):

    prompt,docs = build_prompt(question)

    response = client.models.generate_content(

        model=config.GEMINI_MODEL,

        contents=prompt

    )

    return response.text,docs

In [83]:
ask("Explain Retrieval Augmented Generation")

QUESTION
Explain Retrieval Augmented Generation

ANSWER
* RAG stands for Retrieval Augmented Generation.

DOCUMENTS USED
- sample.txt


In [84]:
# =============================================================================
# CELL 74 : TOTAL LATENCY
# =============================================================================

start = time.time()

ask("Explain Gemini")

print(

    f"Latency : {time.time()-start:.2f} sec"

)

QUESTION
Explain Gemini

ANSWER
Gemini is Google's family of AI models.

DOCUMENTS USED
- sample.txt
Latency : 2.43 sec


In [85]:
# =============================================================================
# CELL 75 : STATISTICS
# =============================================================================

question = "Explain RAG"

queries = generate_queries(question)

print("Generated Queries")

for q in queries:

    print("-",q)

print()

docs = retrieve_multi_query(question)

print("Retrieved")

print(len(docs))

Generated Queries
- Explain RAG
- Explain RAG
- What is RAG
- RAG definition
- RAG explained
- Retrieval Augmented Generation meaning

Retrieved
12
